[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/humanization/humanization_guide/10_ranking_report/10_ranking_lab.ipynb)


# 10 — 후보 랭킹 · 리포트 — 앞 랩 산출물을 하나의 순위로

> 본문 [`10_ranking_report.md`](10_ranking_report.md) 와 **한 절씩 짝지어** 보세요.
> **실측 소요 —** Sapiens 재스코어링(후보 5종) **2초 미만** · ANARCI germline(10 체인) **0.42초**
> **앞 랩에서 이어져요** — Ch.05~Ch.09 의 `my_run/` 을 먼저 찾고, 없으면 커밋된 `data/` 로 대신합니다.

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

이 셀이 저장소를 클론하고 `10_ranking_report/` 로 이동해 필요한 패키지만 깝니다(로컬에서 `10_ranking_report/` 안에 열었다면 클론 생략).
끝나면 **`my_run/`**(내가 만들 결과)과 **`data/`**(커밋된 레퍼런스)가 준비돼요 — 랩은 `my_run/` 을 먼저 찾고 없으면 `data/` 로 폴백하며 어느 쪽을 썼는지 매번 print 합니다.
- ANARCI 는 numbering 을 **hmmscan(HMMER)** 서브프로세스로 돌려요. Colab 에서는 `apt-get install -y hmmer` 가 함께 실행돼요 — pip 만으로는 `hmmscan` 이 없어 죽어요.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "10_ranking_report"
APT_PKGS = "hmmer"     # Colab 에서만: 시스템 패키지 (hmmer = ANARCI 가 부르는 hmmscan)
PIP_PKGS = "pandas matplotlib pyyaml sapiens anarci abnumber"     # 없는 것만 설치

import os, sys, json, time, shutil, pathlib, subprocess, importlib, importlib.util
IN_COLAB = "google.colab" in sys.modules

# 라이브러리 내부 소음 끄기 — 아래 두 줄은 결과와 무관한데 셀 출력 첫머리를 덮어요.
#   · tqdm "IProgress not found"  : 진행바를 위젯 대신 글자로 그린다는 안내일 뿐
#   · pkg_resources is deprecated : lightning/torch 가 남긴 setuptools 경고
# 진단이 필요하면 이 두 줄을 지우고 다시 실행하세요.
import warnings
warnings.filterwarnings("ignore", message=".*IProgress not found.*")
warnings.filterwarnings("ignore", message=".*pkg_resources is deprecated.*")
warnings.filterwarnings("ignore", message=".*Bio.pairwise2 has been deprecated.*")
# 가중치 로딩 진행바와 transformers 안내문도 결과를 밀어내요 (import 전에 꺼 둡니다)
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
import logging
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)   # "unauthenticated requests" 안내

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막아요.
# (멈춤은 예외가 안 나서 폴백이 안 걸려요 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어져요)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, check=True, timeout=None, quiet=False):
    # timeout 을 꼭 주세요 — 네트워크가 '멈춘 채' 매달리면 예외가 안 나서 data/ 폴백이 안 걸립니다.
    """quiet=True 면 출력을 삼키고 **실패했을 때만** 보여줘요.
    apt-get 은 "(Reading database ... 5%(Reading database ... 10%" 같은 진행률을 600자 넘게 쏟아내는데,
    그게 노트북을 연 학습자가 보는 첫 화면을 덮어버려서 실패로 오해하게 만들거든요."""
    print("$", cmd)
    if not quiet:
        return subprocess.run(cmd, shell=True, check=check, timeout=timeout)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if p.returncode != 0:
        print((p.stdout or "") + (p.stderr or ""))
        if check:
            raise subprocess.CalledProcessError(p.returncode, cmd)
    return p

_MARK = "humanization_viz.py"          # 이 파일이 있는 폴더 = 가이드 루트

def _find_root():
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "가이드 루트를 못 찾았어요. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

GUIDE_ROOT = ROOT.resolve()
os.chdir(GUIDE_ROOT / CHAPTER)         # data/ 상대경로가 그대로 동작
sys.path.insert(0, str(GUIDE_ROOT))    # humanization_viz import
sys.path.insert(0, str(GUIDE_ROOT / CHAPTER))

_IMPORT_NAME = {"biopython": "Bio", "pyyaml": "yaml", "scikit-learn": "sklearn"}

def _have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p.replace("-", "_")))]
    if miss:
        print("설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))
        importlib.invalidate_caches()

_APT = APT_PKGS.split() if (APT_PKGS and IN_COLAB) else []   # 모아서 apt 를 한 번만 돌립니다
if PIP_PKGS:
    _ensure(PIP_PKGS)

def _ensure_pkg_resources():
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 합니다.
    if importlib.util.find_spec("pkg_resources") is None:
        _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
        importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    _APT.append("fonts-nanum")             # Colab 엔 한글 폰트가 없어 라벨이 □ 로 깨져요

if _APT:                                   # apt 인덱스 갱신은 한 번만 (ANARCI 는 hmmscan 이 필요해요)
    _run("apt-get -qq update", timeout=600, quiet=True)
    _run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y " + " ".join(_APT), timeout=900, quiet=True)


# ── ① 내가 만든 결과(my_run) 우선 · ② 없으면 커밋된 레퍼런스(data/) ─────────────
MY  = pathlib.Path("my_run"); MY.mkdir(exist_ok=True)
REF = pathlib.Path("data")

def find_one(pattern, ref=REF, quiet=False):
    """산출물 하나 찾기 — 내가 만든 my_run/ 을 먼저 뒤지고, 없으면 커밋된 레퍼런스에서."""
    hits = sorted(MY.rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과]   {hits[0]}")
        return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"'{pattern}' 을 my_run/ 에서도 {ref}/ 에서도 찾지 못했어요."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def find_prev(chapter, pattern, ref=REF, quiet=False):
    """앞 챕터에서 직접 만든 산출물 → 없으면 이 챕터 data/ 레퍼런스."""
    hits = sorted((GUIDE_ROOT / chapter / "my_run").rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과 · {chapter}] {hits[0]}")
        return hits[0]
    return find_one(pattern, ref, quiet)

def read_fasta(path):
    out, name, buf = {}, None, []
    for line in pathlib.Path(path).read_text().splitlines():
        if line.startswith(">"):
            if name: out[name] = "".join(buf)
            name, buf = line[1:].strip(), []
        elif line.strip():
            buf.append(line.strip())
    if name: out[name] = "".join(buf)
    return out

def write_fasta(path, records):
    pathlib.Path(path).write_text("".join(f">{k}\n{v}\n" for k, v in records.items()))
    return pathlib.Path(path)

PARENTAL = read_fasta(REF / "parental.fasta")      # 가이드 전체를 관통하는 러닝 예제
VH = PARENTAL["parental_H"]
VL = PARENTAL["parental_L"]

print("가이드 루트 :", GUIDE_ROOT)
print("작업 폴더   :", pathlib.Path.cwd())
print(f"러닝 예제   : VH {len(VH)} aa · VL {len(VL)} aa (mouse hybridoma 가정)")

## 1) 직접 실행 — 앞 랩 산출물 모으기 (본문 10.2)

후보 하나를 고르려면 Ch.05~09 가 만든 조각이 **한자리에** 있어야 해요. 본문 10.2 가 프로젝트를 문서가 아니라 DB 로 굴리라고 하는 이유가 이거예요 — 3개월 뒤에 "이 점수가 어디서 나왔더라" 를 다시 묻지 않으려고요.

그래서 이 랩의 첫 규칙은 **모든 값에 출처를 붙이는 것**이에요. 내가 직접 만든 `my_run/` 이 있으면 그걸 쓰고, 없는 것만 커밋된 `data/` 로 채워요. 아래 출력의 왼쪽이 후보, 오른쪽이 그 후보가 어디서 왔는지예요.

In [ ]:
import re, difflib, subprocess, tempfile
import pandas as pd, numpy as np

PREV = [
    ("05_humanize_sapiens", "sapiens_humanized_noguard.fasta", ("sapiens_humanized_VH", "sapiens_humanized_VL"), "sapiens"),
    ("06_cdr_safe_tools",   "humatch_humanised.fasta",         ("humatch_humanised_VH", "humatch_humanised_VL"), "humatch"),
    ("06_cdr_safe_tools",   "anthroab_best_score.fasta",       ("anthroab_bestscore_VH", "anthroab_bestscore_VL"), "anthroab"),
    ("06_cdr_safe_tools",   "anthroab_masked_FRonly.fasta",    ("anthroab_masked_VH", "anthroab_masked_VL"), "anthroabFRmasked"),
]

cands, SRC = {"parental": (VH, VL)}, {"parental": "data/parental.fasta"}
for chapter, fname, keys, label in PREV:
    p = GUIDE_ROOT / chapter / "my_run" / fname
    if p.exists():
        f = read_fasta(p)
        cands[label], SRC[label] = (f[keys[0]], f[keys[1]]), f"내 결과 · {chapter}/my_run/{fname}"

need = [lab for *_, lab in PREV if lab not in cands]      # 빠진 후보만 레퍼런스로 채워요
if need:
    vp = find_one("variants.fasta", quiet=True)
    v = read_fasta(vp)
    for lab in need:
        cands[lab], SRC[lab] = (v[f"{lab}_VH"], v[f"{lab}_VL"]), f"레퍼런스 · {vp}"

for k in cands:
    print(f"  {k:18s} ← {SRC[k]}")
print(f"\n후보 {len(cands)}종 · VH/VL 길이", ", ".join(f"{k} {len(h)}/{len(l)}" for k, (h, l) in cands.items()))
print("길이가 parental(120/111)과 다른 후보는 indel 이 들어간 거라 위치 번호를 그대로 비교하면 안 돼요 (2절에서 처리).")

## 2) 직접 실행 — 랭킹에 쓸 지표 계산 (본문 10.1.1)

본문 10.1.1 이 요구하는 축은 6개예요. 이 랩이 각 축을 무엇으로 채우는지 먼저 못 박아요.

| 본문 10.1.1 축 | 이 랩의 측정값 | 어디서 |
|---|---|---|
| Humanness | Sapiens 재스코어링 paired (정의 b) | 이 셀에서 직접 계산 (후보 5종 2초 미만) |
| Nativeness | AbNatiV1 VH `overall_score` | Ch.07 산출물 → 없으면 `data/` |
| CDR/structure 보존 | CDR-H3 backbone RMSD (Å) | Ch.08 산출물 → 없으면 `data/` |
| Developability | parental 대비 **신규** liability 모티프 수 | 이 셀에서 Ch.09 방식으로 계산 |
| Germline 일관성 | ANARCI V identity **VH·VL 평균** | 이 셀에서 직접 실행 (10 체인 0.42초) |
| 전문가 수동 review | 사람 판단 — 노트북이 만들 수 없어요 | 결측으로 둬요 (3절) |

Ab-RoBERTa naturalness 도 함께 읽지만 **가중합에는 넣지 않아요.** Ch.07 에서 확인했듯 사람다움과 자연스러움은 다른 축이고, Ab-RoBERTa 는 주 humanness 지표가 아니라 이상치 탐지 보조라서요 — 표에는 참고 열로만 둡니다.

germline 은 **VH·VL 평균**으로 통일해요. VH 만 보면 parental 이 0.63 이지만 VL 은 0.81 이라, 한쪽만 쓰면 경쇄를 거의 안 건드린 후보가 부당하게 낮게 나와요.

In [ ]:
# (1) humanness — Sapiens 재스코어링(정의 b), paired 는 길이가중 평균
hum, hum_src = {}, ""
try:
    import sapiens

    def mean_self_prob(seq, chain):
        m = sapiens.predict_scores(seq, chain)
        return float(np.mean([m.loc[i, aa] for i, aa in enumerate(seq)]))

    t0 = time.time()
    for n, (h, l) in cands.items():
        ph, pl = mean_self_prob(h, "H"), mean_self_prob(l, "L")
        hum[n] = (ph * len(h) + pl * len(l)) / (len(h) + len(l))
    hum_src = f"직접 계산 · Sapiens 재스코어링 {time.time()-t0:.1f}초"
except Exception as e:
    hp = find_one("humanness_all_candidates.csv", quiet=True)
    ref = pd.read_csv(hp)
    ref = ref[ref.chain == "paired"].set_index("candidate")["mean_self_prob"]
    hum = {n: float(ref[n]) if n in ref.index else np.nan for n in cands}
    hum_src = f"레퍼런스 · {hp} (Sapiens 실패 — {type(e).__name__})"

# (2) germline V identity — ANARCI 직접 실행, VH·VL 따로 받아 평균
gvh, gvl, germ_src = {}, {}, ""
t0 = time.time()
with tempfile.TemporaryDirectory() as td:
    td = pathlib.Path(td)
    write_fasta(td / "all.fa", {f"{n}__{c}": s for n, (h, l) in cands.items()
                                for c, s in (("VH", h), ("VL", l))})
    try:
        r = subprocess.run(["ANARCI", "-i", str(td / "all.fa"), "-s", "imgt", "--csv",
                            "-o", str(td / "gl"), "--assign_germline", "--use_species", "human"],
                           capture_output=True, text=True, timeout=600)   # 멈춘 채 매달리지 않게
        rc = r.returncode
    except (FileNotFoundError, subprocess.TimeoutExpired):
        rc = 127                       # ANARCI/hmmscan 이 없거나 600초 초과 → 레퍼런스로
    got = {}
    if rc == 0:
        for f in sorted(td.glob("gl_*.csv")):
            df = pd.read_csv(f)
            if "v_identity" not in df.columns:
                continue               # ANARCI 판올림으로 컬럼명이 바뀌면 레퍼런스로 넘어가요
            for _, row in df.iterrows():
                n, ch = str(row["Id"]).split("__")
                got[(n, ch)] = float(row["v_identity"])
    if len(got) == 2 * len(cands):
        gvh = {n: got[(n, "VH")] for n in cands}
        gvl = {n: got[(n, "VL")] for n in cands}
        germ_src = f"직접 실행 · ANARCI --assign_germline {len(got)} 체인 {time.time()-t0:.2f}초"
    else:
        gp = find_one("germline_all_candidates.csv", quiet=True)
        g = pd.read_csv(gp)
        gvh = {n: float(g[(g.candidate == n) & (g.chain == "VH")]["v_identity"].iloc[0]) for n in cands if (g.candidate == n).any()}
        gvl = {n: float(g[(g.candidate == n) & (g.chain == "VL")]["v_identity"].iloc[0]) for n in cands if (g.candidate == n).any()}
        germ_src = f"레퍼런스 · {gp}"

# (3) nativeness — Ch.07 의 AbNatiV 셀은 체크포인트 33분/2.6GB 때문에 기본 비활성이고,
#     켜도 결과가 모델별 abnativ/*_seq_scores.csv 로 흩어져 이 챕터가 읽는 합본 형식이 아니에요.
abn_dir = GUIDE_ROOT / "07_nativeness" / "my_run"
abn_merged = sorted(abn_dir.rglob("abnativ_summary_all_models.csv"))
abn_raw = sorted(abn_dir.glob("abnativ/*_seq_scores.csv"))
if abn_merged:
    abn_path, abn_src = abn_merged[0], f"내 결과 · {abn_merged[0]}"
else:
    abn_path = find_one("abnativ_summary_all_models.csv", quiet=True)
    abn_src = f"레퍼런스 · {abn_path}"
    if abn_raw:
        abn_src += f" (Ch.07 의 raw 산출물 {len(abn_raw)}개는 모델별로 나뉜 형식이라 합본으로 못 읽어요)"
abn = pd.read_csv(abn_path)
a1 = abn[(abn.model_generation == "AbNatiV1") & (abn.variant.str.endswith("_VH"))]
nat = {str(r.variant).split("_")[0]: float(r.overall_score) for r in a1.itertuples()}

# (4) naturalness — 참고 열
abr_path = find_prev("07_nativeness", "abroberta_scores_summary.csv", quiet=True)
abr = pd.read_csv(abr_path)
ntr = abr[abr.chain == "paired"].set_index("variant")["mean_logp"].to_dict()

# (5) CDR 보존 · (6) liability 신규 — Ch.04 CDR 표, Ch.09 정규식
ct_path = find_prev("04_sequence_qc", "cdr_table_imgt.csv", quiet=True)
ct = pd.read_csv(ct_path)
CDR_H3 = str(ct[(ct.chain == "H") & (ct.cdr == "CDR3")]["sequence"].iloc[0])
MOTIFS = {"N-glycosylation": r"N[^P][ST]", "deamidation": r"N[GS]",
          "isomerization": r"DG", "oxidation": r"[MW]"}

def scan(seq):
    """서열 → {모티프: 1-based 위치 집합}"""
    return {m: {x.start() + 1 for x in re.finditer(p, seq)} for m, p in MOTIFS.items()}

par_scan = {"VH": scan(VH), "VL": scan(VL)}

# (7) 구조 — Ch.08 이 접은 후보만 값이 있어요
rm_path = find_prev("08_structure", "cdr_h3_rmsd_summary.csv", quiet=True)
rm = pd.read_csv(rm_path)
h3_rmsd = float(rm[rm.metric.str.startswith("cdr_h3")]["value_angstrom"].iloc[0])
fw_rmsd = float(rm[rm.metric.str.startswith("framework")]["value_angstrom"].iloc[0])

rows = []
for n, (h, l) in cands.items():
    lost = [f"{r['chain']}-{r['cdr']}" for _, r in ct.iterrows()
            if str(r["sequence"]) not in (h if r["chain"] == "H" else l)]
    new_g = len(scan(h)["N-glycosylation"] - par_scan["VH"]["N-glycosylation"]) \
          + len(scan(l)["N-glycosylation"] - par_scan["VL"]["N-glycosylation"])
    new_all = sum(len(scan(h)[m] - par_scan["VH"][m]) + len(scan(l)[m] - par_scan["VL"][m])
                  for m in MOTIFS)
    rows.append({"candidate": n,
                 "humanness": round(hum.get(n, np.nan), 4),
                 "nativeness_AbNatiV1_VH": round(nat[n], 4) if n in nat else np.nan,
                 "germline_V_VH": gvh.get(n, np.nan), "germline_V_VL": gvl.get(n, np.nan),
                 "germline_V_mean": round(np.mean([gvh[n], gvl[n]]), 4) if n in gvh and n in gvl else np.nan,
                 "CDR_kept": 6 - len(lost), "CDR_lost": ";".join(lost) or "-",
                 "CDR_H3_kept": CDR_H3 in h,
                 "new_glyc": new_g, "new_liabilities": new_all,
                 "naturalness_AbRoBERTa": round(ntr[n], 4) if n in ntr else np.nan,
                 "cdr_h3_rmsd": 0.0 if n == "parental" else (h3_rmsd if n == "sapiens" else np.nan)})

mt = pd.DataFrame(rows).set_index("candidate")
mt.to_csv(MY / "metrics_table.csv")
print("[출처]")
print("  humanness  ←", hum_src)
print("  germline   ←", germ_src)
print("  nativeness ←", abn_src)
print("  naturalness ←", abr_path)
print("  CDR 표     ←", ct_path)
print("  구조 RMSD  ←", rm_path)
print("→", MY / "metrics_table.csv")
display(mt)

nan_ax = {k: list(mt.index[mt[k].isna()]) for k in
          ("nativeness_AbNatiV1_VH", "cdr_h3_rmsd", "germline_V_mean", "humanness")
          if mt[k].isna().any()}
print()
for k, v in nan_ax.items():
    print(f"  결측 — {k}: {', '.join(v)}")
print(f"판정 — 5후보 × 6축 중 실제로 채워진 칸은 {int(mt[['humanness','nativeness_AbNatiV1_VH','germline_V_mean','new_liabilities','cdr_h3_rmsd']].notna().sum().sum())}/25 개예요.")
print("빈 칸은 '나쁨' 이 아니라 '측정 안 함' 이에요. 3절에서 이 둘을 다르게 다뤄요.")

## 3) 직접 실행 — 가중합 랭킹과 결측 처리 (본문 10.1.1 · 10.2)

가중치는 본문 10.1.1 값을 그대로 씁니다.

```
Final score = 0.25 humanness + 0.20 nativeness + 0.20 CDR/structure 보존
            + 0.15 developability + 0.10 germline 일관성 + 0.10 전문가 review
```

문제는 **빈 칸**이에요. 본문 10.2 가 못 박듯 `null` 은 0 이 아니에요 — "측정 안 함" 과 "나쁨" 은 다른 상태고, 빈 칸을 0 으로 메우면 그 축의 가중치를 **통째로 감점**으로 바꿔요. 그래서 결측 축은 **분모에서 빼고**, 남은 축으로 가중치를 다시 100% 가 되게 나눠요.

```
score = Σ(w_k · norm_k) / Σ(w_k)     ← 둘 다 '값이 있는 축' 에 대해서만
```

축을 0~1 로 접는 방법도 축마다 달라요.

| 축 | 접는 법 | 왜 |
|---|---|---|
| humanness · nativeness · germline | 후보 5종 min-max | 상대 비교용 점수라 절대 기준선이 없어요 |
| CDR/structure 보존 | `1 − RMSD / 2 Å` (0~1 clip) | RMSD 는 절대 스케일이 있어요. 측정값이 parental·Sapiens 둘뿐이라 min-max 를 쓰면 0/1 플래그가 돼요 |
| developability | `(최대 − 신규건수) / 최대` | 신규 0건이 곧 만점이라 0 에 앵커를 걸어요 |
| 전문가 review | 접지 않아요 | 사람 판단이라 이 노트북이 채울 수 없어요 → 전 후보 결측 |

In [ ]:
W = {"humanness": 0.25, "nativeness": 0.20, "structure": 0.20,
     "developability": 0.15, "germline": 0.10, "expert_review": 0.10}

def minmax(s):
    """값 있는 칸만 0~1 로. 전부 같으면 0.5, 결측은 결측 그대로 둬요."""
    s = s.astype(float)
    ok = s.notna()
    if ok.sum() == 0 or s[ok].max() == s[ok].min():
        return s.where(~ok, 0.5)
    return (s - s[ok].min()) / (s[ok].max() - s[ok].min())

def build_norm(tbl):
    nm = pd.DataFrame(index=tbl.index)
    nm["humanness"]  = minmax(tbl["humanness"])
    nm["nativeness"] = minmax(tbl["nativeness_AbNatiV1_VH"])
    nm["germline"]   = minmax(tbl["germline_V_mean"])
    nm["structure"]  = (1 - tbl["cdr_h3_rmsd"].astype(float) / 2.0).clip(0, 1)
    mx = float(tbl["new_liabilities"].max())
    nm["developability"] = 1.0 if mx == 0 else (mx - tbl["new_liabilities"]) / mx
    nm["expert_review"] = np.nan          # 사람 판단 — 결측으로 남겨요
    return nm[list(W)]

def weighted_score(nm):
    num = sum((nm[k] * w).fillna(0.0) for k, w in W.items())
    den = sum(nm[k].notna() * w for k, w in W.items())
    return (num / den).round(4), den.round(2), (num / sum(W.values())).round(4)

norm = build_norm(mt)
score, used_w, score_if_zero = weighted_score(norm)

rank = mt.copy()
rank["score"] = score
rank["used_weight"] = used_w
rank["missing_axes"] = [", ".join(k for k in W if pd.isna(norm.loc[i, k])) for i in norm.index]
out = rank.sort_values("score", ascending=False)
display(out[["score", "used_weight", "missing_axes", "humanness", "nativeness_AbNatiV1_VH",
             "germline_V_mean", "new_liabilities", "cdr_h3_rmsd"]])

print("[정규화 후 축별 점수 — 빈 칸이 결측]")
display(norm.round(3))

delta = (score - score_if_zero).round(4)
print("결측을 0 으로 메웠다면 점수가 어떻게 달라지나")
display(pd.DataFrame([
    {"후보": n,
     "재정규화 (이 랩 방식)": round(float(score[n]), 4),
     "0 대입 (흔한 실수)": round(float(score_if_zero[n]), 4),
     "차이": f"{delta[n]:+.4f}",
     "사용 가중치": f"{used_w[n]:.2f} / 1.00"}
    for n in out.index]))
gap = float(out["score"].iloc[0] - out["score"].iloc[1])
worst = delta.drop(index=out.index[0]).idxmax()
print(f"\n판정 — 0 대입은 결측이 많은 후보를 최대 {delta.max():.4f} 깎아요({worst} 가 가장 크게 손해).")
if delta.max() > gap:
    print(f"이 감점폭은 1·2위 격차 {gap:.4f} 보다 커요 — 0 으로 메웠다면 순위가 뒤집혔을 크기예요.")
else:
    print(f"이번엔 1·2위 격차 {gap:.4f} 보다 작아 순서는 유지되지만, 점수는 근거 없이 낮아져요.")
print("측정하지 않은 축 때문에 후보가 떨어지면 그건 후보의 문제가 아니라 데이터의 구멍이에요 — 채워서 다시 재요.")
low = out.index[out["used_weight"] < 0.7]
print("사용 가중치 0.70 미만(근거가 얇아 점수를 그대로 믿기 어려운 후보) —",
      ", ".join(low) if len(low) else "없음")

## 4) 직접 실행 — hard filter (본문 10.1.2)

가중합은 **치명적 결함을 묻어요.** CDR-H3 가 깨져 결합이 사라진 후보도 humanness 가 높으면 총점 1위에 오를 수 있거든요. 그래서 점수와 **별개로** 무조건 걸리는 선을 둬요.

본문 10.1.2 의 항목은 7개인데, 이 랩의 환경에서 **실제로 잴 수 있는 건 그중 일부**예요. 못 재는 항목을 조용히 빼면 "통과" 가 실제보다 후해지니까, 무엇을 재고 무엇을 못 재는지 먼저 표로 박아요.

In [ ]:
FILTERS = pd.DataFrame([
    {"본문 10.1.2 항목": "CDR-H3 핵심 residue 변경", "이 랩": "측정",
     "근거": "Ch.04 CDR 표의 CDR-H3 문자열이 후보 VH 에 그대로 있는지"},
    {"본문 10.1.2 항목": "알려진 paratope residue 변경", "이 랩": "못 잼",
     "근거": "이 프로젝트에 paratope 목록이 없어요 (본문 10.2 YAML 도 known_paratope_residues: [])"},
    {"본문 10.1.2 항목": "VH/VL interface core residue 변경", "이 랩": "못 잼",
     "근거": "interface core 잔기 정의와 접촉 분석을 이 가이드에서 돌리지 않았어요"},
    {"본문 10.1.2 항목": "CDR 구조 RMSD 급증", "이 랩": "부분",
     "근거": f"Ch.08 이 접은 후보만 값이 있어요 (기준 1.0 Å · framework 자체는 {fw_rmsd:.4f} Å)"},
    {"본문 10.1.2 항목": "CDR/paratope 신규 N-glycosylation", "이 랩": "측정",
     "근거": "Ch.09 정규식 N[^P][ST] · 사슬 전체 기준 (CDR 국한 판정은 Ch.09 4절)"},
    {"본문 10.1.2 항목": "severe hydrophobic/charge patch 증가", "이 랩": "못 잼",
     "근거": "구조 기반 SAP·charge patch 미실행 (Ch.09 5절에서 미실행으로 정리)"},
    {"본문 10.1.2 항목": "AbNatiV/OASis 가 parental 대비 미개선", "이 랩": "측정",
     "근거": "AbNatiV1 VH overall_score 를 parental 과 비교. 값이 없는 후보는 Sapiens humanness 로 대신 재고 표에 표기"},
])
display(FILTERS)

par_nat = nat.get("parental", np.nan)
par_hum = float(mt.loc["parental", "humanness"])
RMSD_CUT = 1.0

flags, basis = {}, {}
for n in mt.index:
    r = mt.loc[n]
    f = []
    if not bool(r["CDR_H3_kept"]):
        f.append("CDR-H3 변경")
    if int(r["CDR_kept"]) < 6:
        f.append(f"CDR 파손 {6 - int(r['CDR_kept'])}개({r['CDR_lost']})")
    if int(r["new_glyc"]) > 0:
        f.append(f"신규 N-glyc {int(r['new_glyc'])}건")
    if pd.notna(r["cdr_h3_rmsd"]) and float(r["cdr_h3_rmsd"]) >= RMSD_CUT:
        f.append(f"CDR-H3 RMSD {float(r['cdr_h3_rmsd']):.3f} Å ≥ {RMSD_CUT} Å")
    if n in nat and pd.notna(par_nat):
        basis[n] = "AbNatiV1 VH"
        if float(r["nativeness_AbNatiV1_VH"]) <= par_nat:
            f.append("nativeness 미개선")
    else:
        basis[n] = "Sapiens humanness(AbNatiV 결측 대체)"
        if float(r["humanness"]) <= par_hum:
            f.append("humanness 미개선 · AbNatiV 결측 대체 판정")
    flags[n] = "; ".join(f) or "pass"
flags["parental"] = "(baseline)"

out["hard_filter"] = [flags[n] for n in out.index]
out["filter_basis"] = [basis[n] for n in out.index]
out[["score", "used_weight", "missing_axes", "hard_filter", "filter_basis", "humanness",
     "nativeness_AbNatiV1_VH", "germline_V_mean", "new_liabilities", "cdr_h3_rmsd",
     "CDR_kept", "CDR_lost", "naturalness_AbRoBERTa"]].to_csv(MY / "ranking.csv")
display(out[["score", "hard_filter", "filter_basis", "CDR_kept", "CDR_lost", "new_glyc", "cdr_h3_rmsd"]])
print("→", MY / "ranking.csv")

adv = out[out.hard_filter == "pass"]
print("hard filter 통과 —", ", ".join(adv.index) if len(adv) else "없음")
top = out.index[0]
if top != "parental" and flags[top] != "pass":
    print(f"판정 — 점수 1위는 {top}({out.loc[top, 'score']:.4f}) 인데 hard filter 에 걸려요 [{flags[top]}].")
    print("가중합이 왜 결론이 될 수 없는지가 여기서 드러나요 — 점수는 평균이고, 결합은 평균으로 안 붙어요.")
if len(adv):
    pick = adv.index[0]
    print(f"실험으로 넘길 후보는 {pick} 예요 (통과 후보 중 점수 최상위 {out.loc[pick, 'score']:.4f}).")
else:
    print("통과 후보가 없으면 backmutation 을 걸어 다시 돌려요 — 걸린 항목을 되돌리는 게 다음 라운드의 목표예요.")
print(f"단, 못 잰 항목이 {int((FILTERS['이 랩'] == '못 잼').sum())}종 남아 있어요. "
      "'pass' 는 '이 랩이 잰 범위에서 통과' 라는 뜻이지 무결이 아니에요.")

## 5) 그림 — 최종 순위와 축별 기여 (본문 10.1.1 · 10.2)

표만 보면 "왜 이 후보가 1위인지" 가 안 보여요. 점수를 **축별로 쪼개 쌓은 막대**로 그리면, 어느 축이 그 후보를 밀어 올렸는지가 한눈에 들어와요.

여기서 제일 조심할 게 **결측**이에요. 빈 칸을 0 으로 그리면 "재지 않은 축" 이 "0 점인 축" 처럼 보여서, 3절에서 재정규화로 바로잡은 걸 그림이 도로 망가뜨려요. 그래서 그림도 두 가지를 분리해요.

| 그림 요소 | 뜻 |
|---|---|
| 색칠된 구간 | 실제로 측정한 축의 기여 `w × 정규화값` |
| **빗금 구간** | 결측 축을 분모에서 빼서 되돌아온 몫 = `재정규화 점수 − 0 대입 점수`. **0 으로 메웠다면 사라지는 부분** |
| 오른쪽 격자 | 후보 × 축 커버리지. 빗금 + `미측정` 이 결측 칸이에요 |
| 빨간 후보 이름 | hard filter 에 걸린 후보(4절 결과) |

In [ ]:
import humanization_viz          # import 만으로 한글 폰트가 등록돼요(안 하면 제목·축이 □ 로 깨져요)
import matplotlib.pyplot as plt
from IPython.display import Image, display

AX_COLOR = {"humanness": "#3f6fb5", "nativeness": "#5aa2c9", "structure": "#7bbf6a",
            "developability": "#e0a63c", "germline": "#c1666b", "expert_review": "#9b8ec4"}
AX_LABEL = {k: f"{k} ({W[k]:.2f})" for k in W}

ordr = list(out.index)[::-1]                 # 아래→위 = 꼴찌→1위 (막대는 위가 1위로 보이게)
yy   = np.arange(len(ordr))

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13.6, 0.95 * len(ordr) + 2.8),
                               gridspec_kw={"width_ratios": [2.3, 1.15]})

left = np.zeros(len(ordr))
for k in W:                                   # ① 측정한 축만 쌓아요 — 결측은 아예 안 그려요(0 으로 안 메움)
    v = np.array([0.0 if pd.isna(norm.loc[n, k]) else float(norm.loc[n, k]) * W[k] for n in ordr])
    axL.barh(yy, v, left=left, height=0.62, color=AX_COLOR[k],
             edgecolor="white", linewidth=0.7, label=AX_LABEL[k])
    left += v

# ② 결측 축을 분모에서 뺀 덕분에 되돌아온 몫 — 0 대입이면 사라지는 부분이라 빗금으로 따로 그려요
boost = np.array([float(score[n] - score_if_zero[n]) for n in ordr])
axL.barh(yy, boost, left=left, height=0.62, color="#f4f4f4", edgecolor="#8c8c8c",
         hatch="///", linewidth=0.9, label="결측 축 재정규화 몫 (0 대입이면 사라짐)")

for i, n in enumerate(ordr):
    axL.text(float(score[n]) + 0.012, yy[i], f"{score[n]:.4f}", va="center", ha="left",
             fontsize=10, fontweight="bold")

hf = out["hard_filter"].to_dict()
axL.set_yticks(yy)
axL.set_yticklabels([f"{n}\n(사용 가중치 {used_w[n]:.2f})" for n in ordr], fontsize=9)
for t, n in zip(axL.get_yticklabels(), ordr):
    if hf.get(n) not in ("pass", "(baseline)"):
        t.set_color("#b03030")                # hard filter 탈락 = 빨간 이름
axL.set_xlim(0, max(0.9, float(score.max()) + 0.16))
axL.set_xlabel("final score = Σ(w · 정규화값) / Σ(측정된 w)")
axL.set_title("최종 순위 — 축별 기여 누적 (빨간 이름 = hard filter 탈락)", fontweight="bold")
axL.grid(axis="x", alpha=0.25); axL.set_axisbelow(True)
axL.legend(fontsize=8, loc="lower right", framealpha=0.95)

AXES = list(W)                                # ③ 커버리지 격자 — 어느 후보의 어느 축이 비었는지
for i, n in enumerate(ordr):
    for j, k in enumerate(AXES):
        v = norm.loc[n, k]
        if pd.isna(v):
            axR.add_patch(plt.Rectangle((j - 0.46, i - 0.40), 0.92, 0.80, facecolor="#f4f4f4",
                                        edgecolor="#8c8c8c", hatch="///", linewidth=0.9))
            axR.text(j, i, "미측정", ha="center", va="center", fontsize=7.5, color="#555555",
                     bbox={"facecolor": "white", "edgecolor": "none", "pad": 1.2})
        else:
            axR.add_patch(plt.Rectangle((j - 0.46, i - 0.40), 0.92, 0.80, facecolor=AX_COLOR[k],
                                        alpha=0.22 + 0.66 * float(v), edgecolor="white", linewidth=0.9))
            axR.text(j, i, f"{float(v):.2f}", ha="center", va="center", fontsize=8.5, color="#101010")
axR.set_xlim(-0.5, len(AXES) - 0.5); axR.set_ylim(-0.6, len(ordr) - 0.4)
axR.set_xticks(range(len(AXES))); axR.set_xticklabels(AXES, fontsize=8, rotation=30, ha="right")
axR.set_yticks(yy); axR.set_yticklabels(ordr, fontsize=9)
axR.set_title("축 커버리지 — 빗금 = 측정 안 함 (0 점이 아님)", fontweight="bold")
for s in ("top", "right", "left", "bottom"):
    axR.spines[s].set_visible(False)
axR.tick_params(length=0)

fig.tight_layout()
png = MY / "10_ranking.png"; fig.savefig(png, dpi=150, bbox_inches="tight")
display(Image(str(png)))          # 저장만 하면 셀에 안 보여요 — 반드시 표시까지
plt.close(fig)
print("→", png)

In [ ]:
miss = {n: [k for k in W if pd.isna(norm.loc[n, k])] for n in out.index}
print("점수를 '실제로 잰 몫' 과 '재정규화로 채운 몫' 으로 쪼개 보면")
display(pd.DataFrame([
    {"후보": n,
     "score": round(float(score[n]), 4),
     "측정분": round(float(score_if_zero[n]), 4),
     "재정규화분": round(float(score[n] - score_if_zero[n]), 4),
     "결측 축": " · ".join(miss[n]) or "없음"}
    for n in out.index]))
mostmiss = max(miss, key=lambda n: len(miss[n]))
gainv    = {n: float(score[n] - score_if_zero[n]) for n in out.index}
bigg     = max(gainv, key=gainv.get)
print(f"\n판정 — 빗금은 '0 점' 이 아니라 '재지 않은 축' 이에요. 축이 가장 많이 빈 후보는 {mostmiss}"
      f" ({len(miss[mostmiss])}축 결측 · 사용 가중치 {used_w[mostmiss]:.2f}) 라, 오른쪽 격자의 빗금 칸이 제일 많아요.")
print(f"       빗금 막대가 가장 넓은 후보는 {bigg} ({gainv[bigg]:.4f}) — 0 으로 메웠다면 이만큼이 그대로 감점이 됐을 몫이에요.")
print("       색칠 구간이 0 인 칸(격자 0.00)은 '재긴 쟀는데 후보 중 꼴찌' 라는 뜻이라 빗금과 전혀 다른 상태예요 — 둘을 같은 색으로 그리면 안 돼요.")
print("       빗금이 넓을수록 점수의 근거가 얇다는 뜻이라, 순위를 그대로 믿지 말고 그 축부터 채워야 해요.")
print("       빨간 이름은 점수와 무관하게 hard filter 에 걸린 후보예요 — 막대가 길어도 실험으로 넘기지 않아요.")

## 6) 직접 실행 — candidate report · GuideDB (본문 10.1.3 · 10.2)

후보 하나를 실험 담당자에게 넘기는 최소 단위가 본문 10.1.3 의 한 장짜리 표예요. 마지막 줄은 반드시 **advance / backmutate / reject** 중 하나로 끝나야 해요.

같은 내용을 본문 10.2 의 GuideDB YAML 로도 떨어뜨려요. 두 형식의 차이는 하나예요 — CSV 는 사람이 읽고, YAML 은 다음 라운드가 읽어요. 그래서 YAML 에서는 측정하지 않은 값을 **`null` 그대로** 남겨요.

In [ ]:
import yaml

def mutations(par, cand):
    """parental raw 1-based 치환 목록. 길이가 다르면(indel) 정렬로 이어 붙여 세요."""
    if len(par) == len(cand):
        return [f"{a}{i+1}{b}" for i, (a, b) in enumerate(zip(par, cand)) if a != b], 0
    muts, indel = [], 0
    for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, par, cand, autojunk=False).get_opcodes():
        if op == "replace":
            for k in range(min(i2 - i1, j2 - j1)):
                muts.append(f"{par[i1+k]}{i1+k+1}{cand[j1+k]}")
        if op in ("replace", "insert", "delete"):
            indel += abs((i2 - i1) - (j2 - j1))
    return muts, indel

report_rows = []
for n in out.index:
    if n == "parental":
        continue
    ch, cl = cands[n]
    mh, ih = mutations(VH, ch)
    ml, il = mutations(VL, cl)
    r = out.loc[n]
    report_rows.append({
        "Candidate ID": f"HZ_{n}_01", "Method": n,
        "VH mutations": f"{len(mh)}개" + (f" +indel {ih}" if ih else "") + " — " + ", ".join(mh[:6]) + ("…" if len(mh) > 6 else ""),
        "VL mutations": f"{len(ml)}개" + (f" +indel {il}" if il else "") + " — " + ", ".join(ml[:6]) + ("…" if len(ml) > 6 else ""),
        "CDR mutations": "none (6/6 보존)" if int(r.CDR_kept) == 6 else f"{6-int(r.CDR_kept)}개 파손 — {r.CDR_lost}",
        "Humanness (paired)": r.humanness,
        "AbNatiV1 VH": r["nativeness_AbNatiV1_VH"],
        "Germline V (VH·VL 평균)": r["germline_V_mean"],
        "Structure": "미측정" if pd.isna(r.cdr_h3_rmsd) else f"CDR-H3 RMSD {float(r.cdr_h3_rmsd):.4f} Å",
        "Developability": "clean" if int(r.new_liabilities) == 0 else f"신규 liability {int(r.new_liabilities)}건(N-glyc {int(r.new_glyc)})",
        "Final score": r.score, "Used weight": r.used_weight,
        "Recommendation": "advance" if r.hard_filter == "pass" else f"reject/backmutate ({r.hard_filter})",
    })
rep = pd.DataFrame(report_rows)
rep.to_csv(MY / "candidate_report.csv", index=False)
display(rep[["Candidate ID", "CDR mutations", "Humanness (paired)", "Final score", "Used weight", "Recommendation"]])

gl = pd.read_csv(find_one("germline_assignment.csv", quiet=True))
def gene_of(chain, gtype):
    q = gl[(gl.chain == chain) & (gl.gene_type == gtype)]
    return (str(q["gene"].iloc[0]), float(q["identity_pct"].iloc[0])) if len(q) else ("unknown", float("nan"))
hv, hv_id = gene_of("H", "V")
lv, lv_id = gene_of("L", "V")
hj, hj_id = gene_of("H", "J")

def nn(x):
    """측정 안 한 값은 0 이 아니라 null 로 — 본문 10.2 의 규칙."""
    return None if pd.isna(x) else float(x)

db = {
    "project": {"id": "HZ_running_example", "parent_clone": "parental",
                "date": time.strftime("%Y-%m-%d")},
    "input_sequences": {"heavy": {"name": "parental_H", "sequence": VH, "species": "mouse"},
                        "light": {"name": "parental_L", "sequence": VL, "species": "mouse"}},
    "annotation": {
        "numbering_scheme": "IMGT", "cdr_definition": "IMGT",
        "heavy_germline": hv, "heavy_germline_v_identity_pct": round(hv_id, 2),
        "light_germline": lv, "light_germline_v_identity_pct": round(lv_id, 2),
        "heavy_j_germline": hj, "heavy_j_identity_pct": round(hj_id, 2),
        "heavy_j_note": "동점 tie-break 이라 도구마다 갈려요 (Ch.04 참고)",
        "heavy_cdr3": CDR_H3, "known_paratope_residues": [],
    },
    "scoring": {"weights": W, "missing_policy": "결측 축은 분모에서 제외하고 가중치 재정규화 (0 대입 금지)",
                "germline_metric": "ANARCI V identity · VH·VL 평균"},
    "candidates": [
        {"id": f"HZ_{n}_01", "method": n,
         "sequences": {"heavy": cands[n][0], "light": cands[n][1]},
         "mutations": {"heavy": len(mutations(VH, cands[n][0])[0]),
                       "light": len(mutations(VL, cands[n][1])[0])},
         "scores": {"humanness_sapiens_paired": nn(out.loc[n, "humanness"]),
                    "nativeness_abnativ1_vh": nn(out.loc[n, "nativeness_AbNatiV1_VH"]),
                    "naturalness_abroberta_paired": nn(out.loc[n, "naturalness_AbRoBERTa"]),
                    "germline_v_identity_mean": nn(out.loc[n, "germline_V_mean"]),
                    "final_score": nn(out.loc[n, "score"]),
                    "used_weight": nn(out.loc[n, "used_weight"])},
         "structure": {"cdr_h3_rmsd": nn(out.loc[n, "cdr_h3_rmsd"])},
         "developability": {"new_liabilities": int(out.loc[n, "new_liabilities"]),
                            "new_n_glyc": int(out.loc[n, "new_glyc"])},
         "cdr_preserved": f"{int(out.loc[n, 'CDR_kept'])}/6",
         "hard_filter": out.loc[n, "hard_filter"],
         "decision": "advance" if out.loc[n, "hard_filter"] == "pass" else "reject/backmutate"}
        for n in out.index if n != "parental"
    ],
}
(MY / "candidate_report.yaml").write_text(yaml.safe_dump(db, allow_unicode=True, sort_keys=False))
print("→", MY / "candidate_report.csv", "·", MY / "candidate_report.yaml")

_yml = (MY / "candidate_report.yaml").read_text()
print(f"\n아래는 방금 쓴 candidate_report.yaml({len(_yml.splitlines())}줄) 에서 **발췌**한 거예요.")
print("파일에는 후보 {}건이 다 들어 있고, 여기서는 앞머리 설정 블록과 후보 1건만 펼칩니다"
      .format(len(db["candidates"])))
print("(서열 두 줄은 화면만 잡아먹어서 발췌에서 뺐어요 — 파일에는 들어 있습니다).")

print("\n[1/2] annotation · scoring — '무엇을 어떤 규칙으로 쟀는가'")
print(yaml.safe_dump({k: db[k] for k in ("annotation", "scoring")}, allow_unicode=True, sort_keys=False))
print(f"[2/2] candidates 중 1건 ({db['candidates'][0]['id']}) — '그 규칙으로 이 후보가 어떻게 나왔는가'")
print(yaml.safe_dump({"candidates": [{k: v_ for k, v_ in db["candidates"][0].items() if k != "sequences"}]},
                     allow_unicode=True, sort_keys=False))

nulls = sum(1 for cd in db["candidates"] for v_ in list(cd["scores"].values()) + list(cd["structure"].values()) if v_ is None)
print(f"\n판정 — YAML 에 남은 null {nulls}개는 '아직 안 쟀다' 는 기록이에요. 0 으로 메웠다면 이 정보가 사라져요.")
print("다음 라운드는 이 null 을 채우는 것부터 시작해요 — 무엇을 더 재야 하는지가 파일 안에 적혀 있는 셈이에요.")

## 7) 레퍼런스 대조 (본문 10.1.1)

같은 코드에 **커밋된 `data/` 만** 넣어 돌려요. 실행을 건너뛴 사람도 같은 표에 도달하는지, 내 순위가 레퍼런스 순위와 같은지 확인하는 절이에요.

In [ ]:
def ref_file(name):
    """이 절만은 내 결과가 아니라 커밋된 data/ 를 씁니다 — 대조의 기준면이라서요."""
    p = REF / name
    assert p.exists(), f"{p} 가 없어요. 저장소를 얕게 클론했다면 data/ 가 빠졌는지 확인하세요."
    return p

ref_v = read_fasta(ref_file("variants.fasta"))
ref_cands = {n: (ref_v[f"{n}_VH"], ref_v[f"{n}_VL"]) for n in cands}
ref_hum = pd.read_csv(ref_file("humanness_all_candidates.csv"))
ref_hum = ref_hum[ref_hum.chain == "paired"].set_index("candidate")["mean_self_prob"]
ref_gm = pd.read_csv(ref_file("germline_all_candidates.csv")).groupby("candidate")["v_identity"].mean()
ref_abn = pd.read_csv(ref_file("abnativ_summary_all_models.csv"))
ref_abn = ref_abn[(ref_abn.model_generation == "AbNatiV1") & (ref_abn.variant.str.endswith("_VH"))]
ref_nat = {str(r.variant).split("_")[0]: float(r.overall_score) for r in ref_abn.itertuples()}

ref_rows = []
for n, (h, l) in ref_cands.items():
    lost = sum(1 for _, r in ct.iterrows() if str(r["sequence"]) not in (h if r["chain"] == "H" else l))
    new_all = sum(len(scan(h)[m] - par_scan["VH"][m]) + len(scan(l)[m] - par_scan["VL"][m]) for m in MOTIFS)
    ref_rows.append({"candidate": n, "humanness": float(ref_hum.get(n, np.nan)),
                     "nativeness_AbNatiV1_VH": ref_nat.get(n, np.nan),
                     "germline_V_mean": float(ref_gm.get(n, np.nan)),
                     "new_liabilities": new_all, "CDR_kept": 6 - lost,
                     "cdr_h3_rmsd": 0.0 if n == "parental" else (h3_rmsd if n == "sapiens" else np.nan)})
ref_mt = pd.DataFrame(ref_rows).set_index("candidate")
ref_score, ref_w, _ = weighted_score(build_norm(ref_mt))
ref_out = ref_mt.assign(score=ref_score, used_weight=ref_w).sort_values("score", ascending=False)
display(ref_out[["score", "used_weight", "humanness", "nativeness_AbNatiV1_VH",
                 "germline_V_mean", "new_liabilities", "CDR_kept"]].round(4))

mine, refo = list(out.index), list(ref_out.index)
print("내 순위      :", " > ".join(mine))
print("레퍼런스 순위:", " > ".join(refo))
if mine == refo:
    print("판정 — 두 순위가 같아요. 실행을 건너뛴 사람도 같은 결론에 도달해요.")
else:
    diff = [n for n in mine if mine.index(n) != refo.index(n)]
    print(f"판정 — 순위가 다른 후보 {len(diff)}개 ({', '.join(diff)}). 내가 만든 후보 서열이 레퍼런스와 다르면 정상이에요.")
print("두 표의 숫자 자체가 다르면 후보 서열이 다른 것이고, 숫자가 같은데 순위만 다르면 가중치를 바꾼 거예요.")

## 8) in silico 는 여기까지 — 실험으로 넘기기 (본문 10.3 · 10.4)

이 가이드가 답한 건 **"계산상 더 사람답고 그럴듯해졌다"** 까지예요. 실제로 붙는지·안정한지·만들어지는지는 컴퓨터가 단정할 수 없어요. 그래서 랭킹의 결론은 "이게 정답" 이 아니라 **"이 순서로 실험한다"** 예요.

본문 10.3 의 최소 검증 순서에서 앞의 둘은 관문이고, **진짜 판정은 ③ 결합력**에서 나요. humanization 의 제1 실패 모드가 "사람다워졌지만 안 붙는다" 라서, parental 과 humanized 를 같은 조건에서 SPR/BLI 로 재 KD 가 유지되는지부터 봐요. 여기서 무너지면 뒤의 Tm·응집 데이터는 볼 이유가 없어요.

그리고 한 번으로 끝내지 않아요. 본문 10.4 의 **lab-in-the-loop** 은 예측 → wet 검증 → **그 실험 데이터로 모델 재학습** → 다시 예측으로 고리를 닫는 구조예요. Genentech/Roche 사례는 4개 표적(EGFR·IL-6·HER2·OSM)에 **1,800개 넘는 변이체**를 설계·실험하고 **4 라운드**로 표적마다 **3–100× 결합력 향상**(최고 ~100 pM)을 얻었어요. 그 사례는 humanization 이 아니라 affinity maturation 이라, 가져올 건 숫자가 아니라 **루프 구조**예요 — oracle 을 결합력 + humanness/면역원성으로 바꿔 끼우면 그대로 적용돼요.

In [ ]:
WET = pd.DataFrame([
    {"순서": 1, "단계": "소량 발현", "방법": "HEK293/CHO transient", "역할": "관문"},
    {"순서": 2, "단계": "정제", "방법": "Protein A/G · SEC purity", "역할": "관문"},
    {"순서": 3, "단계": "결합력", "방법": "ELISA 또는 BLI/SPR (parental 대비 KD 유지)", "역할": "★1순위 판정"},
    {"순서": 4, "단계": "안정성", "방법": "DSF/Tm · accelerated stability", "역할": "판정"},
    {"순서": 5, "단계": "응집", "방법": "SEC-MALS 또는 DLS", "역할": "판정"},
    {"순서": 6, "단계": "특이성", "방법": "cross-reactivity / polyspecificity panel", "역할": "판정"},
    {"순서": 7, "단계": "기능 assay", "방법": "neutralization · blocking · internalization 등", "역할": "판정"},
])
picks = list(adv.index) if len(adv) else []
WET["대상 후보"] = ", ".join(picks) if picks else "backmutation 후 재평가"
WET["parental 대조"] = ["필요" if s in (1, 2) else "필수" for s in WET["순서"]]
WET.to_csv(MY / "wet_lab_plan.csv", index=False)
display(WET)
print("→", MY / "wet_lab_plan.csv")

print(f"\n판정 — in silico 로 좁힌 후보는 {len(cands)-1}종 중 {len(picks)}종이에요"
      + (f" ({', '.join(picks)})." if picks else "."))
print(f"첫 실험은 3번 결합력이에요 — parental 과 {picks[0] if picks else '재설계 후보'} 를 같은 조건에서 SPR/BLI 로 재고,")
print("KD 가 parental 대비 유지되면 4~7번으로 넘어가요. 무너지면 backmutation 으로 되돌아와요.")
print("그 실험값을 다시 이 표(metrics_table.csv)의 새 열로 붙이면 고리가 닫혀요 — 그게 lab-in-the-loop 이에요.")

## 이 랩에서 확인한 것

1. 랭킹은 **본문 10.1.1 의 가중합 + 10.1.2 의 hard filter** 두 층이에요. 점수는 평균이고, 결합은 평균으로 붙지 않아요.
2. 결측은 **분모에서 빼고 재정규화**해요. 0 으로 메우면 측정하지 않은 축이 그대로 감점이 돼요 — 이 랩에서는 구조 RMSD 가 없는 후보가 **0.21~0.23점** 손해를 보는데, 1·2위 격차보다 큰 값이에요.
3. 실측 지표(후보 5종).
   - **humanness**(Sapiens 재스코어링 paired) — parental 0.7303 · Sapiens **0.8424** · Humatch 0.7988 · AnthroAb 0.7941 · AnthroAb(FR-masked) 0.7136
   - **nativeness**(AbNatiV1 VH) — parental 0.6477 · Sapiens **0.8803** · Humatch 0.8305 · AnthroAb 0.8064 · FR-masked **측정 안 됨**
   - **germline V identity**(VH·VL 평균) — parental **0.72**(VH 0.63 · VL 0.81) → Sapiens 0.835 · Humatch 0.81
   - **CDR 보존** — Humatch **6/6**, AnthroAb(FR-masked) **6/6**, Sapiens **1/6**(CDR-H1 만 보존), AnthroAb(best_score) **1/6**(CDR-L3 만 보존)
   - **구조** — Sapiens CDR-H3 RMSD **0.5406 Å**(framework 0.2707 Å). 나머지 후보는 접지 않아 결측
4. 그래서 **"가장 사람다운 후보"와 "실험으로 넘길 후보"가 달라요.** 점수 1위 Sapiens 는 CDR 5개가 파손돼 hard filter 에 걸리고, 통과하는 건 CDR 을 지킨 Humatch 예요. Sapiens 계열을 쓰려면 Ch.05 의 **CDR 가드 적용본**으로 다시 만들어야 해요.
5. hard filter 7종 중 이 환경에서 잰 건 3종(+구조는 부분)이에요. **paratope 변경 · VH/VL interface core · charge patch** 는 못 쟀으니 `pass` 는 "잰 범위에서 통과" 로 읽어요.
6. 순위 그림(`10_ranking.png`)은 점수를 **축별 기여로 쪼갠 누적 막대**예요. 빗금 구간이 **결측 축을 0 으로 메우지 않아 되돌아온 몫**이라, 그림만 봐도 어느 후보의 점수가 얇은 근거 위에 서 있는지 보여요.
7. 산출물 — `my_run/metrics_table.csv` · `ranking.csv` · `candidate_report.csv` · `candidate_report.yaml`(GuideDB) · `wet_lab_plan.csv` · `10_ranking.png`.

전체 체크리스트·용어집 → **[Ch.11 부록](../11_appendix/11_appendix.md)**